# Extension: Decision Tree
In this notebook, we aim to re-construct the auROC and auPRC plots utilizing the various subsets of the HSA Data: Naive, Full, CDC A, CDC B, CDC CL

## Load Dependencies

In [1]:
#%reset
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn import tree
from sklearn import metrics

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

from sklearn.metrics import accuracy_score, confusion_matrix, matthews_corrcoef
from num2words import num2words
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, KFold, RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score, average_precision_score
import word2number
from word2number import w2n
from sklearn.tree import DecisionTreeClassifier
import pickle
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import RocCurveDisplay
import random
from matplotlib.patches import Polygon
import shap
import os

from Functions import prep_training_test_data_period, prep_training_test_data, calculate_metrics,cross_validation_leave_geo_out, prep_training_test_data_shifted, add_labels_to_subplots, LOOCV_by_HSA_dataset, save_in_HSA_dictionary, prepare_data_and_model
hfont = {'fontname':'Helvetica'}
palette = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#e5c494']
import json 

## Load Data

In [2]:
HSA_weekly_data_all = pd.read_csv("HSA_Weekly_Surveillance.csv")

columns_to_remove = [col for col in HSA_weekly_data_all.columns if 'cases' in col]
HSA_weekly_data_no_cases_no_deaths = HSA_weekly_data_all.drop(columns=columns_to_remove)

columns_to_remove = [col for col in HSA_weekly_data_no_cases_no_deaths.columns if 'deaths' in col]
HSA_weekly_data_no_cases_no_deaths = HSA_weekly_data_no_cases_no_deaths.drop(columns=columns_to_remove)

HSA_weekly_data_all.loc[1, 'week_fifty-two_beds_over_15_100k']

0.0

## Define Global Variables

In [3]:
no_iterations = 100
geography_column = 'HSA_ID'
geo_split = 0.9
time_period = 'period'  # Choose 'period', 'exact', or 'shifted'
size_of_test_dataset = 1
train_weeks_for_initial_model = 1
weeks_to_predict = range(1, 123 - size_of_test_dataset - 3 - train_weeks_for_initial_model)

weeks_in_future = 3
weight_col = 'weight'
keep_output = True

# This is for SHAP Analysis - the order of the column names in the X_test files
feature_names=['COVID-19 admissions', 
               'COVID-19 ICU beds', 
               'COVID-19 hospital beds', 
               'Perc. beds with \nCOVID-19 patients', 
               '\u0394 COVID-19 admissions',
               '\u0394 COVID-19 ICU beds',
               '\u0394 COVID-19 hospital beds',
               '\u0394 Perc. beds with \nCOVID-19 patients',
               '> 15 per 100,000 COVID-19 \npatients in hospital beds']

# Just to check
print(len(feature_names))

9


## Location and County Data

In [4]:
## County Data 
data_by_county = pd.read_csv('county_time_data_all_dates.csv')

data_by_county = data_by_county.dropna(subset=['admits_weekly', 'deaths_weekly', 'cases_weekly', 'icu_weekly', 'beds_weekly', 'perc_covid'])
data_by_county['CTYNAME'] = data_by_county['CTYNAME'].apply(lambda x: x.split()[0])
data_by_county['CTYNAME'] = data_by_county['fips'].astype(str) + '' + data_by_county['CTYNAME']
data_by_county['beds_over_15_100k'] = (data_by_county['beds_weekly'] > 15) * 1

# Redo dates
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i

## DELTA POLYGON 
start_date = pd.to_datetime('2021-06-30')
end_date = pd.to_datetime('2021-10-26')
data_by_county['date'] = pd.to_datetime(data_by_county['date'])
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i
# Find the indices of rows that match the exact start and end dates
matching_indices_start = data_by_county.loc[data_by_county['date'] <= start_date].index.max()
matching_indices_end = data_by_county.loc[data_by_county['date'] <= end_date].index.max()
first_week_delta = data_by_county.loc[matching_indices_start, 'week']
last_week_delta = data_by_county.loc[matching_indices_end, 'week']
start_date = pd.to_datetime('2021-10-26')
end_date = pd.to_datetime('2022-09-27')
data_by_county['date'] = pd.to_datetime(data_by_county['date'])
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i
# Find the indices of rows that match the exact start and end dates
matching_indices_start = data_by_county.loc[data_by_county['date'] <= start_date].index.max()
matching_indices_end = data_by_county.loc[data_by_county['date'] <= end_date].index.max()
first_week_omricon = data_by_county.loc[matching_indices_start, 'week']
last_week_omricon = data_by_county.loc[matching_indices_end, 'week']

## CDC POLYGON 
start_date = pd.to_datetime('2021-03-01')
end_date = pd.to_datetime('2022-01-24')
data_by_county['date'] = pd.to_datetime(data_by_county['date'])
for i, week in enumerate(data_by_county['date'].unique()):
    data_by_county.loc[data_by_county['date'] == week, 'week'] = i
# Find the indices of rows that match the exact start and end dates
matching_indices_start = data_by_county.loc[data_by_county['date'] <= start_date].index.max()
matching_indices_end = data_by_county.loc[data_by_county['date'] <= end_date].index.max()
first_week_CDC = data_by_county.loc[matching_indices_start, 'week']
last_week_CDC = data_by_county.loc[matching_indices_end, 'week']

/var/folders/b1/ts1cmy7n6kg0gzvxmtp8cc_80000gn/T/ipykernel_88391/873074510.py:2: DtypeWarning: Columns (47,48,49,50,51,55,56) have mixed types. Specify dtype option on import or set low_memory=False.
  data_by_county = pd.read_csv('county_time_data_all_dates.csv')


## Adding Percent Exceeding Capacity

In [5]:
percent_exceed_capacity = []

# Iterate through the columns of the DataFrame
for column_name in HSA_weekly_data_all.columns:
    if 'beds_over_15_100k' in column_name:
        # Calculate the sum of the column and append it to the list
        column_sum = HSA_weekly_data_all[column_name].sum() / len(HSA_weekly_data_all[column_name])
        percent_exceed_capacity.append(column_sum)

## Define Hyperparameter Tuning Function

In [6]:
def hyperparameter_train_model_DT(X_train, y_train, sample_weights, decision_tree):   
    # Here are the hyperparamters
    param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 5, 8, 12],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
}
    
    # Now we construct the grid of hyperparamters to train
    grid_search = GridSearchCV(estimator=decision_tree,
                           param_grid=param_grid,
                           scoring='average_precision',  # Optimize for F1-score
                           cv=5,
                           n_jobs=-1, # This utilizes all cores in the CPU
                           verbose=1) 
    # Now evaluate
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    
    # Now we return the best model for usage
    best_model = grid_search.best_estimator_
    
    return best_model

## Preparing Training Function

In [7]:
def prep_training_test_data(
    data, no_weeks, weeks_in_future, geography, weight_col, keep_output
):
    ## Get the weeks for the x and y datasets
    x_weeks = []
    y_weeks = []
    for week in no_weeks:
        test_week = int(week) + weeks_in_future
        x_weeks.append("_" + num2words(week) + "_")
        y_weeks.append("_" + num2words(test_week) + "_")
    X_data = pd.DataFrame()
    y_data = pd.DataFrame()
    weights_all = pd.DataFrame()
    missing_data = []
    ## Now get the training data
    k = 0
    for x_week in x_weeks:
        y_week = y_weeks[k]
        k += 1
        weeks_x = [col for col in data.columns if x_week in col]
        columns_x = [geography] + weeks_x + [weight_col]
        data_x = data[columns_x]
        weeks_y = [col for col in data.columns if y_week in col]
        columns_y = [geography] + weeks_y
        data_y = data[columns_y]
        # ensure they have the same amount of data
        # remove rows in test_data1 with NA in test_data2
        data_x = data_x.dropna()
        data_x = data_x[data_x[geography].isin(data_y[geography])]
        # remove rows in test_data2 with NA in test_data1
        data_y = data_y.dropna()
        data_y = data_y[data_y[geography].isin(data_x[geography])]
        data_x = data_x[data_x[geography].isin(data_y[geography])]
        data_x_no_HSA = len(data_x[geography].unique())
        missing_data.append(
            (
                (len(data[geography].unique()) - data_x_no_HSA)
                / len(data[geography].unique())
            )
            * 100
        )
        # get weights
        # weights = weight_data[weight_data[geography].isin(data_x[geography])][[geography, weight_col]]
        X_week = data_x.iloc[:, 1 : len(columns_x)]  # take away y, leave weights for mo
        y_week = data_y.iloc[:, -1]
        y_week = y_week.astype(int)
        weights = X_week.iloc[:, -1]
        if keep_output:
            X_week = X_week.iloc[
                :, : len(X_week.columns) - 1
            ]  # remove the weights and leave "target" for that week
            # rename columns for concatenation
            X_week.columns = range(1, len(data_x.columns) - 1)
        else:
            X_week = X_week.iloc[
                :, : len(X_week.columns) - 2
            ]  # remove the weights and  "target" for that week
            X_week.columns = range(
                1, len(data_x.columns) - 2
            )  # remove the weights and  "target" for that week
            # rename columns for concatenation
        y_week.rename("0", inplace=True)
        X_data = pd.concat([X_data, X_week], axis = 0)
        y_data = pd.concat([y_data, y_week], axis = 0)
        weights.rename("0", inplace=True)
        weights_all = pd.concat([weights_all, weights], axis = 0)
    X_data.reset_index(drop=True, inplace=True)
    y_data.reset_index(drop=True, inplace=True)
    weights_all.reset_index(drop=True, inplace=True)
    return (X_data, y_data, weights_all, missing_data)

## Preparing Folder for the Results

In [8]:
# Preparing folders for the different models
dt_folder = "WIP"

# Double check in-case folders are not prepared
def create_directory_if_not_exists(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f"Created directory: {directory_path}")
    else:
        print(f"Directory already exists: {directory_path}")

# Now apply to the folder names
create_directory_if_not_exists(dt_folder)

Directory already exists: WIP


## Prepare Data (Naive/Full/Reduced/CDC A/ CDC B)

In [10]:
# Naive 
columns_to_select = HSA_weekly_data_all.filter(regex="HSA|beds_over_15_100k|weight").columns.tolist()
naive = HSA_weekly_data_all[columns_to_select]

# Reduced 
reduced_regex_pattern = (
    "HSA|weight|beds_over_15_100k|"
    "admits|icu|beds|perc_covid"
)

columns_to_select_reduced = HSA_weekly_data_all.filter(
    regex=reduced_regex_pattern
).columns.tolist()

columns_to_select_reduced = [col for col in columns_to_select_reduced if "cases" not in col and "deaths" not in col] # Making sure we do not have the cases

reduced = HSA_weekly_data_all[columns_to_select_reduced]

# CDC Optimized
cdc_optimized_pattern = ("HSA|weight|cases|admits|perc_covid|beds_over")
columns_to_select_cdc = HSA_weekly_data_all.filter(
    regex=cdc_optimized_pattern
).columns.tolist()
columns_to_select_cdc = [col for col in columns_to_select_cdc if "delta" not in col]
cdc = HSA_weekly_data_all[columns_to_select_cdc]

# Full
full_regex_pattern = (
    "HSA|weight|beds_over_15_100k|"
    "cases|deaths|admits|icu|beds|perc_covid"
)

columns_to_select_full = HSA_weekly_data_all.filter(
    regex=full_regex_pattern
).columns.tolist()

full = HSA_weekly_data_all[columns_to_select_full]

subsets_HSA = [naive, reduced, full, cdc]

## Loop - Decision Tree

In [12]:
size_of_test_dataset = 1
names = ["Naive", "Reduced", "Full", "CDC"]
count = 0
for subsets in subsets_HSA: # Cycle through the different datasets
    ROC_by_week_dt_period = []
    PRC_by_week_dt_period = [] 
    sensitivity_by_week_dt_period = []
    specificity_by_week_dt_period = []
    ppv_by_week_dt_period = []
    npv_by_week_dt_period = []
    accuracy_by_week_dt_period = []
    norm_MCC_by_week_dt_period = []
    for prediction_week in weeks_to_predict:
        print(prediction_week)
        print(range(1 , int(prediction_week + train_weeks_for_initial_model) + 1))
        print(range(int(prediction_week + train_weeks_for_initial_model) + 1, int(prediction_week + train_weeks_for_initial_model + size_of_test_dataset) + 1))
    
        X_train_dt, y_train_dt, weights_dt, missing_data_train_HSA = prep_training_test_data(subsets, no_weeks=range(1, int(prediction_week + train_weeks_for_initial_model) + 1), weeks_in_future=3, geography='HSA_ID', weight_col='weight', keep_output=True)
    
        X_test_dt, y_test_dt, weights_test_dt, missing_data_test_HSA = prep_training_test_data(subsets, no_weeks=range(int(prediction_week + train_weeks_for_initial_model) + 1, int(prediction_week + train_weeks_for_initial_model + size_of_test_dataset) + 1), weeks_in_future=3, geography='HSA_ID', weight_col='weight', keep_output=True)
        
        weights_dt = weights_dt.to_numpy()
        weights_dt = weights_dt.ravel()
        
        # Instantiate classifier
        clf_dt =  DecisionTreeClassifier(random_state=10, class_weight='balanced') # 
        
        # Hyperparamter train
        best_clf_dt = hyperparameter_train_model_DT(X_train_dt, y_train_dt, weights_dt, clf_dt)        
        
        # Make predictions on the test set
        y_pred = best_clf_dt.predict(X_test_dt)
        y_pred_proba = best_clf_dt.predict_proba(X_test_dt)
    
        # Evaluate the accuracy of the model
        accuracy_by_week_dt_period.append(accuracy_score(y_test_dt, y_pred))
        if len(np.unique(y_test_dt)) > 1:
        # Calculate ROC AUC score only if there are multiple classes
            ROC_by_week_dt_period.append(roc_auc_score(y_test_dt, y_pred_proba[:, 1]))
            PRC_by_week_dt_period.append(average_precision_score(y_test_dt, y_pred_proba[:, 1]))
        else:
            ROC_by_week_dt_period.append(np.nan)
            PRC_by_week_dt_period.append(np.nan)
        conf_matrix = confusion_matrix(y_test_dt, y_pred)
    
        sensitivity, specificity, ppv, npv = calculate_metrics(conf_matrix)
        sensitivity_by_week_dt_period.append(sensitivity)
        specificity_by_week_dt_period.append(specificity)
    
        ppv_by_week_dt_period.append(ppv)
        npv_by_week_dt_period.append(npv)
    
        norm_MCC_by_week_dt_period.append((matthews_corrcoef(y_test_dt, y_pred) + 1)/2)
        
    # Organize the file names - change to represent the proper dataset
    dt_performance_name = f"{names[count]}_dt_performance_stats.pkl"
    
    # Organize into a dictionary
    dt_performance_stats = {
        "ROC": ROC_by_week_dt_period,
        "PRC": PRC_by_week_dt_period,
        "accuracy": accuracy_by_week_dt_period,
        "sensitivity": sensitivity_by_week_dt_period,
        "specificity": specificity_by_week_dt_period,
        "ppv": ppv_by_week_dt_period,
        "npv": npv_by_week_dt_period,
        "MCC": norm_MCC_by_week_dt_period
    }
    
    # Open
    with open(dt_performance_name, 'wb') as f:
        pickle.dump(dt_performance_stats, f)
    
    count += 1

1
range(1, 3)
range(3, 4)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
2
range(1, 4)
range(4, 5)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
3
range(1, 5)
range(5, 6)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
4
range(1, 6)
range(6, 7)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
5
range(1, 7)
range(7, 8)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
6
range(1, 8)
range(8, 9)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
7
range(1, 9)
range(9, 10)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
8
range(1, 10)
range(10, 11)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
9
range(1, 11)
range(11, 12)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
10
range(1, 12)
range(12, 13)
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
11
range(1, 13)
range(13, 14)
Fitting 5 folds for each of 270 candidates, totalling 1350 

# Community Levels
Utilizing the framework produced by the CDC, we extend our analysis.

## Load Datasets

In [13]:
def determine_covid_outcome_indicator(new_cases_per_100k, new_admits_per_100k, percent_beds_100k):
    # ensure numeric and handle NaN defensively
    try:
        c = float(new_cases_per_100k)
        a = float(new_admits_per_100k)
        p = float(percent_beds_100k)
    except Exception:
        return 'Medium'  

    if c < 200:
        if (a >= 20) or (p >= 0.15):
            return 'High'
        elif (a >= 10) or (p >= 0.10):
            return 'Medium'
        else:
            return 'Low'
    else:  # c >= 200
        if (a >= 10) or (p >= 0.10):
            return 'High'
        else:
            return 'Medium'

In [22]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

try:
    from num2words import num2words
    _HAS_NUM2WORDS = True
except Exception:
    _HAS_NUM2WORDS = False

def _week_word_fragment(week):
    wnum = str(int(week))
    frags = [re.escape(wnum)]  # numeric form

    if _HAS_NUM2WORDS:
        wword = num2words(int(week)).lower().strip()
        parts = re.split(r'[\s\-\_]+| and ', wword)
        parts = [p for p in parts if p]  # drop empties
        frag = r'[-_\s]*'.join([re.escape(p) for p in parts])
        frags.append(frag)
        frags.append(re.escape('_'.join(parts)))
        frags.append(re.escape('-'.join(parts)))

    # combine alternatives
    combined = r'(?:' + r'|'.join(frags) + r')'
    return combined

def _find_week_category_col(cols, week, category_keyword):
    wk_frag = _week_word_fragment(week)
    cat_pat = re.escape(category_keyword.lower())
    for c in cols:
        s = str(c).lower()
        if 'delta' in s:
            continue
        if 'week' not in s:
            continue
        if not re.search(cat_pat, s):
            continue
        # now check week fragment as regex
        if re.search(wk_frag, s):
            return c
    return None

def compute_CL_direct(all_weekly_data,
                      weeks_to_predict=range(1, 36),
                      weeks_in_future=3,
                      geography='HSA_ID'):
    preds_rows = []
    metrics = []

    cols = list(all_weekly_data.columns)

    # find geography label in original columns (case-insensitive)
    geo_label = None
    for c in cols:
        if str(c).lower() == geography.lower():
            geo_label = c
            break
    if geo_label is None:
        raise KeyError(f"Could not find geography column named '{geography}' in data. Columns: {cols}")

    for week in weeks_to_predict:
        target_week = week + weeks_in_future
        print(f"Processing week {week} -> target week {target_week}")

        # find indicator columns for this week
        case_col = _find_week_category_col(cols, week, 'case')
        admit_col = _find_week_category_col(cols, week, 'admit')
        perc_col = _find_week_category_col(cols, week, 'perc')  # matches perc, percent, perc_covid

        # find target (future) columns for outcome calculation
        case_col_y = _find_week_category_col(cols, target_week, 'case')
        admit_col_y = _find_week_category_col(cols, target_week, 'admit')
        perc_col_y = _find_week_category_col(cols, target_week, 'perc')

        missing = []
        if not case_col:
            missing.append('cases')
        if not admit_col:
            missing.append('admits')
        if not perc_col:
            missing.append('perc')
        if missing:
            raise KeyError(f"Could not find indicator columns for week {week}: missing {missing}. Available cols: {cols}")

        if not (case_col_y and admit_col_y and perc_col_y):
            print(f"Warning: target columns for week {target_week} not fully found. Found: case_y={case_col_y}, admit_y={admit_col_y}, perc_y={perc_col_y}")

        # subset relevant columns
        subset_cols = [geo_label, case_col, admit_col, perc_col]
        y_cols = []
        if case_col_y:
            y_cols.append(case_col_y)
        if admit_col_y:
            y_cols.append(admit_col_y)
        if perc_col_y:
            y_cols.append(perc_col_y)
        # create dataframe
        df_ind = all_weekly_data[subset_cols].copy()
        # standardize names
        df_ind.columns = [geography, 'cases', 'admits', 'perc_covid']

        # build actual dataframe if target cols available
        if len(y_cols) == 3:
            df_y = all_weekly_data[[geo_label, case_col_y, admit_col_y, perc_col_y]].copy()
            df_y.columns = [geography, 'cases_y', 'admits_y', 'perc_covid_y']
        else:
            # create empty with same index so we can align safely
            df_y = pd.DataFrame({geography: all_weekly_data[geo_label]})
            df_y['cases_y'] = np.nan
            df_y['admits_y'] = np.nan
            df_y['perc_covid_y'] = np.nan

        try:
            maxp = df_ind['perc_covid'].max(skipna=True)
            if pd.notna(maxp) and maxp > 1.5:
                df_ind['perc_covid'] = df_ind['perc_covid'] / 100.0
            maxp_y = df_y['perc_covid_y'].max(skipna=True)
            if pd.notna(maxp_y) and maxp_y > 1.5:
                df_y['perc_covid_y'] = df_y['perc_covid_y'] / 100.0
        except Exception:
            pass

        merged = pd.merge(df_ind, df_y, on=geography, how='left', suffixes=('', '_y'))

        # drop rows where the indicator (cases/admits/perc) are missing
        merged = merged.dropna(subset=['cases', 'admits', 'perc_covid'])

        if merged.shape[0] == 0:
            print(f"Week {week}: no rows with non-missing indicators; skipping.")
            metrics.append({'week': week, 'auroc': np.nan, 'auprc': np.nan, 'n_samples': 0, 'pct_missing': 100.0})
            continue

        # compute predicted CL label using indicator columns (week w)
        merged['pred_label'] = merged.apply(
            lambda r: determine_covid_outcome_indicator(r['cases'], r['admits'], r['perc_covid']),
            axis=1
        )
        merged['pred_binary'] = merged['pred_label'].map({'Low': 0, 'Medium': 0, 'High': 1}).astype(float)

        # compute actual CL label using target columns (week w + weeks_in_future)
        merged['actual_label'] = merged.apply(
            lambda r: determine_covid_outcome_indicator(r['cases_y'], r['admits_y'], r['perc_covid_y'])
            if (pd.notna(r['cases_y']) and pd.notna(r['admits_y']) and pd.notna(r['perc_covid_y']))
            else np.nan,
            axis=1
        )
        merged['actual_binary'] = merged['actual_label'].map({'Low': 0, 'Medium': 0, 'High': 1}).astype(float)

        # drop rows where actual is missing (can't compute metrics without actual)
        valid_mask = merged['actual_binary'].notna()
        pred_bin = merged.loc[valid_mask, 'pred_binary']
        actual_bin = merged.loc[valid_mask, 'actual_binary']

        # compute metrics defensively
        if len(pred_bin) == 0:
            auroc = np.nan
            auprc = np.nan
            note = 'no target values for this week'
        else:
            unique_actual = np.unique(actual_bin)
            if len(unique_actual) < 2:
                auroc = np.nan
                auprc = np.nan
                note = 'single-class actual'
            else:
                try:
                    auroc = roc_auc_score(actual_bin, pred_bin)
                    auprc = average_precision_score(actual_bin, pred_bin)
                    note = ''
                except Exception as e:
                    auroc = np.nan
                    auprc = np.nan
                    note = f'metric error: {e}'

        # record predictions (only rows with indicators present)
        preds_out = merged[[geography]].copy()
        preds_out['week'] = week
        preds_out['pred_label'] = merged['pred_label']
        preds_out['pred_binary'] = merged['pred_binary']
        preds_out['actual_label'] = merged['actual_label']
        preds_out['actual_binary'] = merged['actual_binary']

        preds_rows.append(preds_out)

        pct_missing = 100.0 * (1 - (valid_mask.sum() / merged.shape[0]))

        metrics.append({
            'week': week,
            'auroc': auroc,
            'auprc': auprc,
            'n_samples': int(valid_mask.sum()),
            'pct_missing': pct_missing,
            'note': note
        })

        print(f"Week {week}: n_valid={int(valid_mask.sum())}, AUROC={auroc}, AUPRC={auprc}, note={note}")

    all_preds = pd.concat(preds_rows, ignore_index=True) if preds_rows else pd.DataFrame(columns=[geography, 'week', 'pred_label', 'pred_binary', 'actual_label', 'actual_binary'])
    metrics_df = pd.DataFrame(metrics)
    return all_preds, metrics_df

all_preds, metrics_df = compute_CL_direct(HSA_weekly_data_all, weeks_to_predict=range(84, 119), weeks_in_future=3, geography='HSA_ID')

print(metrics_df)

Processing week 84 -> target week 87
Week 84: n_valid=787, AUROC=0.6148934370771313, AUPRC=0.07808571169712047, note=
Processing week 85 -> target week 88
Week 85: n_valid=787, AUROC=0.770097948623175, AUPRC=0.04256353063575486, note=
Processing week 86 -> target week 89
Week 86: n_valid=787, AUROC=0.8898171950871179, AUPRC=0.07773420836502362, note=
Processing week 87 -> target week 90
Week 87: n_valid=787, AUROC=0.8295787545787546, AUPRC=0.076946057965753, note=
Processing week 88 -> target week 91
Week 88: n_valid=787, AUROC=0.4910828025477707, AUPRC=0.0025412960609911056, note=
Processing week 89 -> target week 92
Week 89: n_valid=787, AUROC=0.4942381562099872, AUPRC=0.007623888182973317, note=
Processing week 90 -> target week 93
Week 90: n_valid=787, AUROC=0.5736931027628701, AUPRC=0.05793317229149504, note=
Processing week 91 -> target week 94
Week 91: n_valid=787, AUROC=0.5625, AUPRC=0.14278907242693772, note=
Processing week 92 -> target week 95
Week 92: n_valid=787, AUROC=0.5

In [23]:
filename = "CCL_performance_stats.pkl"

dt_performance_stats = {
    "ROC": metrics_df["auroc"],
    "PRC": metrics_df["auprc"]
}

with open(filename, 'wb') as f:
    pickle.dump(dt_performance_stats, f)